# A Tour of psnek

This notebook demonstrates the essentials of psnek: wrapping loops, manual and nested bars, unknown totals, taller bars, failures, and parallel and async bars.

The panel and its tables are covered in the [panel notebook](02_panel_tables.ipynb), and resuming and plain logs in the [resilience notebook](04_resilience.ipynb).

In [1]:
import asyncio
import math
import time
from collections.abc import AsyncIterator, Callable, Iterable, Iterator

from IPython.display import HTML, display

import psnek

psnek.__version__

'0.1.0'

## Every State at a Glance

`psnek.gallery()` renders every state of the bar from fixed snapshots. Every state other than running is named in text as well as color.

In [2]:
display(HTML(psnek.gallery(width=100, html=True)))

## Wrapping a Loop Costs Almost Nothing

`track()` runs no Python code per item, so its overhead is tiny even on a loop that does almost nothing, which is the worst case. Each timing below is the best of five runs, with and without a bar.

In [3]:
ITEMS = 5_000_000
REPEATS = 5


def best_time(make_items: Callable[[], Iterable[int]], desc: str) -> float:
    """Return the fastest of `REPEATS` runs of a loop summing ``make_items()``."""
    times = []
    for _ in psnek.track(REPEATS, desc):
        items = make_items()
        start = time.perf_counter()
        total = 0
        for item in items:
            total += item
        times.append(time.perf_counter() - start)
    return min(times)


numbers = list(range(ITEMS))
for label, source in (("range", range(ITEMS)), ("list", numbers)):
    bare = best_time(lambda source=source: source, f"{label}, bare")
    tracked = best_time(
        lambda source=source, label=label: psnek.track(source, f"sum over a {label}"),
        f"{label}, tracked",
    )
    added = (tracked - bare) / ITEMS * 1e9
    print(
        f"{label}: bare {bare:.3f}s, tracked {tracked:.3f}s, {added:+.1f} ns per item"
    )

# range, bare        ##############  100.0%              5/5         0:00    9.8it/s  done
# range, tracked     ##############  100.0%              5/5         0:00    9.6it/s  done
# list, bare         ##############  100.0%              5/5         0:00   13.8it/s  done
# list, tracked      ##############  100.0%              5/5         0:00   11.0it/s  done

range: bare 0.097s, tracked 0.102s, +1.0 ns per item


list: bare 0.068s, tracked 0.088s, +3.8 ns per item


## Manual Bars, Units, and Scales

A `Bar` can be updated manually from any thread. Units and SI or IEC prefixes are set with `Options`.

In [4]:
CHUNK = 4 * 2**20
transfer = psnek.Options(unit="B", scale="iec")
with psnek.Bar(
    total=512 * 2**20, desc="upload model.safetensors", options=transfer
) as bar:
    for _ in range(128):
        time.sleep(0.02)
        bar.update(CHUNK)

# upload model.safetensors ##########  100.0%   512MiB/512MiB         0:03   167MiB/s  done

## Nested Bars and Status Fields

Bars started inside another bar's loop are drawn below it. `status()` adds `key=value` fields after the metrics.

In [5]:
for epoch in psnek.track(3, "epochs"):
    steps = psnek.track(40, f"epoch {epoch + 1}")
    for step in steps:
        time.sleep(0.01)
        steps.status(loss=1 / (1 + epoch * 40 + step), lr=3e-4)

# epochs    #############################  100.0%        3/3         0:01    2.0it/s  done

## Unknown Totals

Without a total, the snake slithers back and forth while the count and rate keep updating.

In [6]:
def stream_records(count: int) -> Iterator[int]:
    """Yield ``count`` records slowly, like a paginated API."""
    for record in range(count):
        time.sleep(0.004)
        yield record


for _ in psnek.track(stream_records(600), "stream records"):
    pass

# stream records ################################        600         0:02    203it/s  done

## Taller Bars

`Options(height=n)` draws a bar over `n` rows. A long description wraps over the rows instead of being cut off, and status fields fill the rows below.

This simulated fine-tuning run updates its description as it moves through epochs and learning-rate phases.

In [7]:
STEPS_PER_EPOCH = 120
EPOCHS = 3
WARMUP_STEPS = 40

two_rows = psnek.Options(height=2, unit="step")
with psnek.Bar(total=EPOCHS * STEPS_PER_EPOCH, options=two_rows) as bar:
    for step in range(EPOCHS * STEPS_PER_EPOCH):
        time.sleep(0.01)
        epoch = step // STEPS_PER_EPOCH + 1
        if step < WARMUP_STEPS:
            phase, scale = "warmup", (step + 1) / WARMUP_STEPS
        else:
            progress = (step - WARMUP_STEPS) / (EPOCHS * STEPS_PER_EPOCH - WARMUP_STEPS)
            phase = "cooldown" if epoch == EPOCHS else "cosine decay"
            scale = 0.1 + 0.45 * (1 + math.cos(math.pi * progress))
        loss = 0.4 + 2.1 * math.exp(-step / 90) + 0.03 * math.sin(step / 3)
        bar.desc = (
            f"fine-tune llama-3-8b on support tickets: epoch {epoch}/{EPOCHS}, {phase}"
        )
        metrics = {"train loss": loss, "ppl": math.exp(loss), "lr": 3e-4 * scale}
        bar.status(metrics)
        bar.update()

# fine-tune llama-3-8b on support  ##########  100.0%    360/360         0:04  81.1step/s  done
  tickets: epoch 3/3, cooldown     ##########  train loss=0.4474  ppl=1.564  lr=3.001e-05

`status()` also accepts a `dict`, whose keys don't need to be valid Python names, and setting a key to `None` removes it. This simulated evaluation shows a `retrying` field only while a slow shard is retried.

In [8]:
RECALL = {"billing": 78, "bug report": 66, "outage": 90}
LABELS = tuple(RECALL)
BATCHES, SHARDS, RETRIED = 240, 8, 5

seen = dict.fromkeys(LABELS, 0)
hits = dict.fromkeys(LABELS, 0)
three_rows = psnek.Options(height=3, unit="batch")
with psnek.Bar(total=BATCHES, options=three_rows) as bar:
    for batch in range(BATCHES):
        shard = batch * SHARDS // BATCHES + 1
        retrying = shard == RETRIED and batch % (BATCHES // SHARDS) < 10
        time.sleep(0.05 if retrying else 0.012)
        label = LABELS[batch % len(LABELS)]
        seen[label] += 1
        hits[label] += (batch * 37) % 100 < RECALL[label]
        bar.desc = f"evaluate ticket classifier on held-out shard {shard}/{SHARDS}"
        metrics = {"accuracy": sum(hits.values()) / (batch + 1)}
        metrics |= {name: hits[name] / seen[name] for name in LABELS if seen[name]}
        metrics["retrying"] = f"shard {shard}" if retrying else None
        bar.status(metrics)
        bar.update()

# evaluate ticket classifier on ##########  100.0%    240/240         0:03  61.3batch/s  done
  held-out shard 8/8            ##########  accuracy=0.7833  bug report=0.6500
                                ##########  billing=0.8000   outage=0.9000

At four rows, the face of a bar without a total grows to match: it still smiles while running, stares when stalled, and winces when stuck. In this simulated crawl, the second site's server goes quiet for a few seconds, long enough for its bar to stall.

In [9]:
SITES = {"docs.python.org": 90, "peps.python.org": 60, "packaging.python.org": 75}
SECTIONS = ("library", "reference", "howto", "tutorial", "whatsnew")
QUIET_SITE, QUIET_PAGE, QUIET_SECONDS = "peps.python.org", 30, 7

four_rows = psnek.Options(height=4)
crawl = psnek.track(SITES, "crawl", options=four_rows)
pages = links = broken = fetched = 0
for site in crawl:
    crawl.desc = f"crawl {len(SITES)} sites for broken links, now on {site}"
    with psnek.Bar(desc=site, options=four_rows) as visit:
        for page in range(SITES[site]):
            quiet = (site, page) == (QUIET_SITE, QUIET_PAGE)
            time.sleep(QUIET_SECONDS if quiet else 0.02)
            found = 7 + page % 13
            pages, links = pages + 1, links + found
            fetched += 14_000 + 97 * page
            broken += page % 17 == 0
            visit.desc = f"{site}/{SECTIONS[page % len(SECTIONS)]}/page-{page:03}.html"
            visit.status(depth=1 + page % 4, links=found)
            visit.update()
            crawl.status(
                pages=pages,
                links=links,
                broken=broken,
                frontier=links - pages,
                mib=f"{fetched / 2**20:.1f}",
            )

# crawl 3 sites for broken links,   ##########  100.0%        3/3         0:12   4.13s/it  done
  now on packaging.python.org       ##########  pages=225   frontier=2659
                                    ##########  links=2884  mib=3.8
                                    ##########  broken=15

Each mood at one, two, and four rows, rendered with `psnek.render(html=True)`:

In [10]:
moods = [
    psnek.Frame(
        n=512,
        elapsed=75,
        rate=6.8,
        desc=f"{state}, {height} {'row' if height == 1 else 'rows'}",
        state=state,
        idle=idle,
        phase=0.8,
        height=height,
    )
    for height in (1, 2, 4)
    for state, idle in (("running", 0.0), ("stalled", 14.0), ("stuck", 61.0))
]
HTML("".join(psnek.render(frame, width=90, html=True) for frame in moods))

## Keeping the Total with `enumerate` and `zip`

An `enumerate()` object has no `len()`, so a bar wrapping one has no percentage or ETA. Wrap the iterable inside it instead. `zip()` stops at its shortest iterable, so wrapping just one of them in `track()` leaves the bar short of its total and ends it as `interrupted`, while `psnek.zip()` takes its total from the shortest iterable and ends as `done`.

In [11]:
shards = [f"shard-{index:03}.parquet" for index in range(150)]
verified = [f"{index * 2654435761 % 2**32:08x}" for index in range(120)]

zero_based, one_based = {}, {}
for index, shard in psnek.track(enumerate(shards), "track(enumerate(shards))"):
    time.sleep(0.004)
    zero_based[shard] = index
for index, shard in enumerate(psnek.track(shards, "enumerate(track(shards))"), 1):
    time.sleep(0.004)
    one_based[shard] = index
unchecked = psnek.track(shards, "zip(track(shards))")
for _shard, _checksum in zip(unchecked, verified, strict=False):
    time.sleep(0.004)
pairs = psnek.zip(shards, verified, desc="psnek.zip(shards, verified)")
for shard, checksum in pairs:
    time.sleep(0.004)
    pairs.status(shard=shard, sha=checksum)

# track(enumerate(shards))    ###########                150         0:00    204it/s  done
# enumerate(track(shards))    ###########  100.0%    150/150         0:00    201it/s  done
! zip(track(shards))          ########*__   80.0%    120/150         0:00    201it/s  interrupted
# psnek.zip(shards, verified) ###########  100.0%    120/120         0:00    201it/s  done

## Failures

Inside `with`, an exception marks the bar as failed. A plain `for` loop can't see the exception, so use `with` when failures matter.

In [12]:
try:
    with psnek.Bar(total=50, desc="flaky job") as bar:
        for step in range(50):
            time.sleep(0.01)
            if step == 31:
                raise ConnectionError("upstream closed the connection")
            bar.update()
except ConnectionError as error:
    print(f"caught: {error}; outcome={bar.outcome}, n={bar.n}")

x flaky job #############________   62.0%      31/50         0:00   78.1it/s  failed ConnectionError

caught: upstream closed the connection; outcome=failed, n=31


## Parallel Map

`pmap()` maps a function over items using threads (or any executor, such as a `ProcessPoolExecutor`), and returns the results in input order.

In [13]:
def fetch(page: int) -> int:
    """Pretend to download ``page`` and return its size in bytes."""
    time.sleep(0.02)
    return page * 1024


sizes = psnek.pmap(fetch, range(400), "fetch pages", workers=16)
print(f"fetched {len(sizes)} pages, {sum(sizes) / 2**20:.1f} MiB")

# fetch pages ###########################  100.0%    400/400         0:00    562it/s  done

fetched 400 pages, 77.9 MiB


## Async Iterables

`atrack()` wraps an async iterable, and top-level `await` works in Jupyter. For concurrent calls, see the [async notebook](05_async.ipynb).

In [14]:
async def messages(count: int) -> AsyncIterator[str]:
    """Yield ``count`` messages from a simulated queue."""
    for index in range(count):
        await asyncio.sleep(0.005)
        yield f"message {index}"


received = [message async for message in psnek.atrack(messages(300), "consume queue")]
print(received[-1])

# consume queue #################################        300         0:01    176it/s  done

message 299
